# Adjustments to the assigned date used

Since 2023 is the year when the source dataset was released and the data after that are supposed to misterpret the information, we decided to re-assign the year after that. Since the adjustments won't affect the data split or LoS predictions training, we simply re-assign the resulted predictions instead of the changing the dates in the  `train_los.csv`.

In [ ]:
import pandas as pd
import os

## Adjustments for `icu_patients`

In [2]:
icu_patients = pd.read_csv('../Merged_Dataset/icu_patients.csv')
icu_patients["intime"] = pd.to_datetime(icu_patients["intime"])
icu_patients["outtime"] = pd.to_datetime(icu_patients["outtime"])
icu_patients["assigned_intime"] = pd.to_datetime(icu_patients["assigned_intime"])
icu_patients["assigned_outtime"] = pd.to_datetime(icu_patients["assigned_outtime"])

In [3]:
print(icu_patients["assigned_intime"].min(), icu_patients["assigned_intime"].max())

2008-01-01 08:01:00 2025-06-23 05:27:15


In [4]:
def assign_year(date):
    """Assign year 2023 to dates in or after 2023."""
    if date.year <= 2023:
        return date
    if date.month==2 and date.day==29:
         return date.replace(year=2023, month=2, day=28)
    return date.replace(year=2023)

In [5]:
icu_patients["assigned_intime"] = icu_patients["assigned_intime"].apply(assign_year)
icu_patients["assigned_outtime"] = icu_patients["assigned_intime"] + pd.to_timedelta(icu_patients["outtime"] - icu_patients["intime"])
print(icu_patients["assigned_intime"].min(), icu_patients["assigned_intime"].max())

2008-01-01 08:01:00 2023-12-31 17:05:36


In [6]:
icu_patients.to_csv('../Merged_Dataset/icu_patients(re-assign).csv', index=False)

## Adjustments of LoS prediction results

In [ ]:
results = pd.read_csv('../Prediction/los_only/Results.csv')
df = results.merge(icu_patients[["subject_id", "hadm_id", "stay_id","intime", "outtime", "assigned_intime"]], on=["subject_id", "hadm_id", "stay_id"], how='left')

In [8]:
df["intime"] = pd.to_datetime(df["intime"])
df["outtime"] = pd.to_datetime(df["outtime"])
df["chartdate"] = pd.to_datetime(df["chartdate"])

df.dropna(subset=["assigned_intime_x"], inplace=True)
df.rename(columns={"assigned_intime_y": "assigned_intime"}, inplace=True)
df["assigned_intime"] = pd.to_datetime(df["assigned_intime"])

df["assigned_chartdate"] = df["assigned_intime"] + pd.to_timedelta(df["chartdate"]-df["intime"])
df.drop(columns=["intime", "outtime"], inplace=True)

In [9]:
df

,subject_id,hadm_id,stay_id,assigned_intime_x,assigned_outtime,chartdate,actual_los,predicted_los,abs_error,assigned_intime,assigned_chartdate
0,17057610,26086792,30045625,2020-05-17 22:24:28,2020-06-16 18:16:36,2200-06-08,8,14.580319,6.580319,2019-05-17 22:24:28,2019-06-08
1,17057610,26086792,30045625,2020-05-17 22:24:28,2020-06-16 18:16:36,2200-06-08,8,14.182450,6.182450,2019-05-17 22:24:28,2019-06-08
2,17057610,26086792,30045625,2020-05-17 22:24:28,2020-06-16 18:16:36,2200-06-08,8,14.969455,6.969455,2019-05-17 22:24:28,2019-06-08
3,17057610,26086792,30045625,2020-05-17 22:24:28,2020-06-16 18:16:36,2200-06-08,8,9.247801,1.247801,2019-05-17 22:24:28,2019-06-08
4,17057610,26086792,30045625,2020-05-17 22:24:28,2020-06-16 18:16:36,2200-06-08,8,9.103970,1.103970,2019-05-17 22:24:28,2019-06-08
...,...,...,...,...,...,...,...,...,...,...,...
242198,17598702,20473158,37693858,2025-06-23 05:27:15,2025-06-27 15:00:40,2144-06-25,2,1.189700,0.810300,2023-06-23 05:27:15,2023-06-25
242199,17598702,20473158,37693858,2025-06-23 05:27:15,2025-06-27 15:00:40,2144-06-25,2,1.001645,0.998355,2023-06-23 05:27:15,2023-06-25
242200,17598702,20473158,37693858,2025-06-23 05:27:15,2025-06-27 15:00:40,2144-06-24,3,1.001645,1.998355,2023-06-23 05:27:15,2023-06-24
242201,17598702,20473158,37693858,2025-06-23 05:27:15,2025-06-27 15:00:40,2144-06-24,3,1.001645,1.998355,2023-06-23 05:27:15,2023-06-24


In [ ]:
df.to_csv('../Prediction/los_only/Results_with_assigned_chartdates(re-assign).csv', index=False)